In [1]:
!pip install qdrant-client fastembed -q
!pip install google-generativeai -q -U

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 337.3/337.3 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.3/105.3 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.3/103.3 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 88.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.8/324.8 kB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 5.6 MB/s eta 0:00:00


In [2]:
import time
import json
import collections
import pandas as pd

from typing import List
from google import genai
from google.colab import userdata
from google.api_core import exceptions
from qdrant_client import QdrantClient, models
from fastembed import TextEmbedding, SparseTextEmbedding

In [3]:
google_client = genai.Client(api_key=userdata.get('GOOGLE_API_KEY'))

In [4]:
import logging
logging.getLogger("tornado.access").setLevel(logging.CRITICAL)

### Load Documents and ground truth data

In [5]:
with open('data/data.json', 'rt') as f_in:
    documents = json.load(f_in)

In [6]:
questions = pd.read_csv('ground_truth.csv')

### Build Qdrant collection

In [7]:
client = QdrantClient(":memory:")

In [8]:
def collection_exists(collection_name: str) -> bool:

    try:
        existing_collections = [col.name for col in client.get_collections().collections]
        return collection_name in existing_collections

    except:
        return False

In [9]:
def build_collection(name: str, vector_config: dict = None, sparse_vector_config: dict = None):

    try:
        vector_config = vector_config or {}
        sparse_vector_config = sparse_vector_config or {}

        exists = collection_exists(name)

        if exists:
            print(f"Collection '{name}' already exists.")
            return

        client.create_collection(
            collection_name = name,
            vectors_config = vector_config,
            sparse_vectors_config = sparse_vector_config
        )

        print(f"Qdrant collection '{name}' created.")

    except Exception as e:
        print(f"Failed to create collection '{name}': {e}")

In [10]:
def populate_collection(name: str, models_names: dict, documents: List[dict]):

    try:
        exists = collection_exists(name)

        if not exists:
            print(f"Collection '{name}' does not exists.")
            return

        points = []

        for record in documents:

            text_embed = f"{record['term']}: {record['definition']} {record['extra']}"

            dict_vector = {}

            for vector_name, model_name in models_names.items():
                dict_vector[vector_name] = models.Document(
                    text = text_embed,
                    model = model_name
                )

            point = models.PointStruct(
                id = record['id'],
                vector = dict_vector,
                payload = {
                    'term': record['term'],
                    'description': f"{record['definition']} {record['extra']}",
                    'models_used': models_names
                }
            )

            points.append(point)

        client.upsert(
            collection_name = name,
            points=points
        )

        print(f"Successfully populated collection '{name}' with {len(points)} records.")

    except Exception as e:
        print(f"An error occurred: {e}")

### Hybrid-reranking search

In [11]:
def rrf_search(question: str, collection_name: str, limit: int = 5):

    if not collection_exists(collection_name):
        print(f"Collection '{collection_name}' does not exist.")
        return

    collection = client.get_collection(collection_name)
    vector_config = collection.config.params.vectors
    sparse_vector_config = collection.config.params.sparse_vectors

    if len(vector_config) + len(sparse_vector_config) < 2:
        print("Both dense and sparse vectors are required for RRF search.")
        return

    sample_point = client.scroll(collection_name=collection_name, limit=1)[0][0]
    used_models = sample_point.payload.get("models_used", {})

    dense = {}
    sparse = {}

    for vector_name, model_name in used_models.items():
        if vector_name in vector_config:
            dense = {"name": vector_name, "model": model_name}
        elif vector_name in sparse_vector_config:
            sparse = {"name": vector_name, "model": model_name}

    if not dense or not sparse:
        print("Dense and sparse vector fields not found in the collection.")
        return

    results = client.query_points(
        collection_name=collection_name,
        query=models.FusionQuery(fusion=models.Fusion.RRF),
        prefetch=[
            models.Prefetch(
                query=models.Document(
                    text=question,
                    model=dense["model"]
                ),
                using=dense["name"],
                limit=2 * limit
            ),
            models.Prefetch(
                query=models.Document(
                    text=question,
                    model=sparse["model"]
                ),
                using=sparse["name"],
                limit=2 * limit
            )
        ],
        limit=limit,
        with_payload=True
    )

    context = {'question': question}

    for i, res in enumerate(results.points):
        context[f'context{i+1}'] = res.payload['description']

    return context

In [12]:
build_collection(
    name = 'hybrid-search-collection',
    vector_config = {
        'dense_text': models.VectorParams(
            size = 512,
            distance = models.Distance.COSINE
        )
    },
    sparse_vector_config = {
        'sparse_text': models.SparseVectorParams(
            modifier = models.Modifier.IDF
        )
    }
)
populate_collection(
    name='hybrid-search-collection',
    models_names={
        'dense_text': 'jinaai/jina-embeddings-v2-small-en',
        'sparse_text': 'Qdrant/bm25'
    },
    documents=documents
)

Qdrant collection 'hybrid-search-collection' created.


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

spanish.txt: 0.00B [00:00, ?B/s]

portuguese.txt: 0.00B [00:00, ?B/s]

turkish.txt:   0%|          | 0.00/260 [00:00<?, ?B/s]

swedish.txt:   0%|          | 0.00/559 [00:00<?, ?B/s]

Successfully populated collection 'hybrid-search-collection' with 829 records.


### Input prompt for Qdrant collection for search

In [13]:
input_prompt = """

    You are given:
    1. A question.
    2. Two context passages (context1 and context2).

    Your task:
    - Answer the question using only the given contexts.
    - If both contexts provide relevant information, combine them into a complete answer.
    - If the contexts are insufficient, explicitly state that the answer cannot be fully determined.
    - Make the response well-structured, clear, and directly connected to the question.

    Input format:

    question: {question}
    context1: {context1}
    context2: {context2}

""".strip()

### Get relevant documents from collection and generate response using LLM

In [14]:
MAX_RETRIES = 5
DELAY_SECONDS = 5

In [15]:
def get_response(question: str, model: str = 'gemini-2.5-flash-lite'):

    context_docs = rrf_search(question=question, collection_name='hybrid-search-collection')

    for attempt in range(MAX_RETRIES):
        try:
            prompt = input_prompt.format(**context_docs)
            response = google_client.models.generate_content(
                model=model,
                contents=prompt
            )

            if hasattr(response, 'text'):
                return response.text
            else:
                print(f"Warning: Response object doesn't have 'text' attribute: {type(response)}")
                return str(response)

        except exceptions.ResourceExhausted as e:
            print(f"Rate limit hit. Attempt {attempt + 1}/{MAX_RETRIES}. Waiting 15 seconds...")
            if attempt < MAX_RETRIES - 1:
                time.sleep(15)

        except exceptions.InternalServerError as e:
            print(f'Internal server error. Attempt {attempt + 1}/{MAX_RETRIES}. Retrying with backoff...')
            if attempt < MAX_RETRIES - 1:
                time.sleep(DELAY_SECONDS * (2 ** attempt))

        except Exception as e:
            print(f"Unexpected error on attempt {attempt + 1}/{MAX_RETRIES}: {e}")
            if attempt < MAX_RETRIES - 1:
                time.sleep(DELAY_SECONDS)

    print(f"Failed to get response after {MAX_RETRIES} attempts for question: {question[:50]}...")
    return None

### Evaluation prompt for response

In [16]:
prompt_evaluate = """

    You will be given:
    1. A question.
    2. A generated answer.

    Your task:
    - Evaluate how relevant the answer is to the question.
    - Use only the information in the given input.

    Output requirements:
    - Assign a relevance score: RELEVANT | PARTIALLY RELEVANT | NOT RELEVANT
    - Provide a brief explanation for your choice.
    - Do not use any code blocks.
    - Use double quotes for all strings, not single quotes
    - Ensure the output is JSON parsable without any additional processing
    - Format your output exactly as shown:

    {{"Relevance": "<your score>", "Explanation": "<your explanation>"}}

    Input:

    question: {question}
    answer: {answer}

""".strip()

### Evaluate generated response with LLM

In [17]:
def get_evaluation(question: str, answer: str):

    dictionary = {'question': question, 'answer': answer}

    response = None

    for attempt in range(MAX_RETRIES):
        try:
            prompt = prompt_evaluate.format(**dictionary)
            response = google_client.models.generate_content(
                model='gemini-2.5-flash-lite',
                contents=prompt
            )
            break

        except exceptions.ResourceExhausted as e:
            print(f"Rate limit hit. Attempt {attempt + 1}/{MAX_RETRIES}. Waiting 15 seconds...")
            if attempt < MAX_RETRIES - 1:
                time.sleep(15)

        except exceptions.InternalServerError as e:
            print(f'Internal server error in evaluation: Trying again..... {attempt + 1}/{MAX_RETRIES}')
            if attempt < MAX_RETRIES - 1:
                time.sleep(DELAY_SECONDS * (2 ** attempt))

        except Exception as e:
            print(f"An unexpected error occurred in evaluation: {e}")
            break

    if response is None:
        print(f"Failed to get evaluation response after {MAX_RETRIES} attempts")
        return None

    try:
        return response.text
    except AttributeError:
        print("Evaluation response object doesn't have 'text' attribute")
        return str(response)

### RAG evaluation on a sample

In [18]:
sample = questions.sample(n=200, random_state=19)

In [20]:
response_25_flash_lite = {}
response_25_flash = {}

In [ ]:
for i in range(len(sample)):

    question = sample.iloc[i]['question']
    question_id = sample.iloc[i]['id']

    if question not in response_25_flash_lite:
        answer = get_response(question=question, model='gemini-2.5-flash-lite')
        response_25_flash_lite[question] = {
            'question_id': question_id,
            'answer': answer
        }

    if question not in response_25_flash:
        answer = get_response(question=question, model='gemini-2.5-flash')
        response_25_flash[question] = {
            'question_id': question_id,
            'answer': answer
        }

In [22]:
len(response_25_flash), len(response_25_flash_lite)

(200, 200)

In [30]:
response_25_flash_dataframe = pd.DataFrame.from_dict(response_25_flash, orient='index')
response_25_flash_dataframe.reset_index(inplace=True)
response_25_flash_dataframe.rename(columns={'index': 'question'}, inplace=True)
response_25_flash_dataframe.to_csv('gemini_flash_25.csv', index=False)

In [31]:
response_25_flash_lite_dataframe = pd.DataFrame.from_dict(response_25_flash_lite, orient='index')
response_25_flash_lite_dataframe.reset_index(inplace=True)
response_25_flash_lite_dataframe.rename(columns={'index': 'question'}, inplace=True)
response_25_flash_lite_dataframe.to_csv('gemini_flash_25_lite.csv', index=False)